# pyfieldml Quickstart

Five-minute demo: install, load a dataset, inspect, **visualize**, evaluate, and export.

## Install

This notebook assumes `pyfieldml` is already installed:

```bash
pip install pyfieldml[meshio,viz]
```

In [ ]:
from pyfieldml import datasets

doc = datasets.load_unit_cube()
print("Version :", doc.source_version)
print("Region  :", doc.region.name)

## What you just loaded

Pyvista renders the FieldML document directly through the `pyfieldml.interop.pyvista.to_pyvista` helper.

In [ ]:
import pyvista as pv

from pyfieldml.interop.pyvista import to_pyvista

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

grid = to_pyvista(doc)
p = pv.Plotter(off_screen=True, window_size=(640, 480), shape=(1, 2))
p.subplot(0, 0)
p.add_mesh(grid, color="skyblue", show_edges=True, edge_color="black")
p.add_text("unit_cube (solid)", font_size=9)
p.view_isometric()
p.show_axes()
p.subplot(0, 1)
p.add_mesh(grid, style="wireframe", color="steelblue", line_width=3)
p.add_text("wireframe", font_size=9)
p.view_isometric()
p.show_axes()
p.show(screenshot="/tmp/quickstart_unit_cube.png")

## Inspect the evaluator graph

In [ ]:
for name, ev in doc.evaluators.items():
    print(f"{name:30s}  {type(ev).__name__}")

## Extract coordinates as NumPy

In [ ]:
coords_ev = doc.evaluators["coordinates"]
xyz = coords_ev.as_ndarray()
print("coords shape:", xyz.shape)
print("coords:")
print(xyz)

## Plot the node cloud in matplotlib

A pure-matplotlib 3D scatter is a minimal way to sanity-check a mesh without any OpenGL stack at all.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], s=80, depthshade=True)
for i, p in enumerate(xyz, start=1):
    ax.text(p[0], p[1], p[2], str(i), fontsize=8)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title(f"unit_cube node cloud (n={xyz.shape[0]})")
fig.tight_layout()

## Evaluate the coordinate field at the element centroid

In [ ]:
coords = doc.field("coordinates")
centroid = coords.evaluate(element=1, xi=(0.5, 0.5, 0.5))
print("centroid:", centroid)

## Export to meshio (requires [meshio] extra)

In [ ]:
try:
    m = doc.to_meshio()
    print("meshio cell type:", m.cells[0].type, "count:", len(m.cells[0].data))
except ImportError as exc:
    print("meshio not installed; install with: pip install pyfieldml[meshio]")
    print(exc)

### Next steps

- See `02_evaluator_graph.ipynb` for a tour of the evaluator hierarchy.
- See `04_muscle_fibers.ipynb` for a fiber-field workflow.
- See `07_real_anatomy.ipynb` for the 10-dataset model-zoo gallery.